# mp4 文字起こし + 前処理 (Colab / GPU)

パイプライン: `mp4` → 音声抽出 → **demucs** ボーカル分離 → 16k mono → **whisperX** (large-v3, 日本語, 単語timestamp + 話者分離) → `json/srt/vtt/txt/tsv`

## 事前準備
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. ローカルの `interfacex/chore/static` を Google Drive にアップロード（mp4 を置く）
3. 話者分離(diarization)を使う場合、HuggingFace token が必要:
   - https://huggingface.co/settings/tokens でトークン作成
   - 以下2モデルの利用規約に同意（要ログイン）:
     - https://huggingface.co/pyannote/speaker-diarization-3.1
     - https://huggingface.co/pyannote/segmentation-3.0

## 1. GPU確認

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'GPU無効: ランタイムのタイプをGPUに変更'
print('CUDA OK:', torch.cuda.get_device_name(0))

## 2. 依存インストール (数分かかる)

In [ ]:
!pip -q install whisperx demucs "transformers>=4.40,<5"
!apt -qq install -y ffmpeg >/dev/null
# whisperx と Colab同梱 transformers の不整合回避のためランタイム再起動
print('インストール完了 -> 自動再起動します。再起動後はこのセルを飛ばしてマウントセル(#3)から実行')
import os; os.kill(os.getpid(), 9)

## 3. Drive マウント + パス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
#@markdown Drive内の static フォルダのパス (mp4が入っている場所)
STATIC_DIR = '/content/drive/MyDrive/chore/static'  #@param {type:'string'}
#@markdown 文字起こしエンジン言語
LANGUAGE = 'ja'  #@param {type:'string'}
#@markdown whisperモデル
MODEL = 'large-v3'  #@param ['large-v3','large-v2','medium','small']
#@markdown 話者分離(diarization)を有効化
USE_DIARIZATION = True  #@param {type:'boolean'}
#@markdown HuggingFace token (diarization使う場合のみ)
HF_TOKEN = ''  #@param {type:'string'}

TRANSCRIPTS_DIR = os.path.join(STATIC_DIR, 'transcripts')
PREPROCESSED_DIR = os.path.join(TRANSCRIPTS_DIR, 'preprocessed')
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

import glob
mp4s = sorted(glob.glob(os.path.join(STATIC_DIR, '*.mp4')))
assert mp4s, f'mp4が見つからない: {STATIC_DIR}'
print(f'{len(mp4s)}本のmp4:')
for p in mp4s: print(' ', os.path.basename(p))

## 4. 前処理関数 (mp4 → demucs vocals → 16k mono)

出力構造（既存ローカルと同形）:
```
transcripts/preprocessed/<name>/source.wav
transcripts/preprocessed/<name>/vocals_16k_mono.wav
transcripts/preprocessed/<name>/demucs/htdemucs/source/{vocals,no_vocals}.wav
```

In [ ]:
import subprocess, shutil

def run(cmd):
    subprocess.run(cmd, check=True)

def preprocess(mp4_path):
    name = os.path.splitext(os.path.basename(mp4_path))[0]
    outdir = os.path.join(PREPROCESSED_DIR, name)
    os.makedirs(outdir, exist_ok=True)
    source_wav = os.path.join(outdir, 'source.wav')
    vocals_16k = os.path.join(outdir, 'vocals_16k_mono.wav')

    if os.path.exists(vocals_16k):
        print(f'[skip] {name} 前処理済')
        return vocals_16k

    # mp4 -> source.wav (44.1k stereo)
    run(['ffmpeg','-y','-i',mp4_path,'-vn','-ac','2','-ar','44100',source_wav])

    # demucs ボーカル分離 (htdemucs, vocals/no_vocals)
    demucs_out = os.path.join(outdir, 'demucs')
    run(['python','-m','demucs','--two-stems=vocals','-n','htdemucs','-o',demucs_out,source_wav])
    vocals = os.path.join(demucs_out,'htdemucs','source','vocals.wav')

    # vocals -> 16k mono (whisper入力)
    run(['ffmpeg','-y','-i',vocals,'-ac','1','-ar','16000',vocals_16k])
    print(f'[ok] {name} 前処理完了')
    return vocals_16k

## 5. 出力ライター (json/srt/vtt/txt/tsv)

In [ ]:
import json

def _ts(sec, comma=True):
    h = int(sec // 3600); m = int((sec % 3600) // 60)
    s = int(sec % 60); ms = int(round((sec - int(sec)) * 1000))
    sep = ',' if comma else '.'
    return f'{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}'

def _spk(seg):
    sp = seg.get('speaker')
    return f'[{sp}] ' if sp else ''

def write_outputs(result, name):
    segs = result['segments']
    base = os.path.join(TRANSCRIPTS_DIR, name)

    with open(base+'.json','w',encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    with open(base+'.txt','w',encoding='utf-8') as f:
        for s in segs:
            f.write(_spk(s) + s['text'].strip() + '\n')

    with open(base+'.srt','w',encoding='utf-8') as f:
        for i, s in enumerate(segs, 1):
            f.write(f'{i}\n{_ts(s["start"])} --> {_ts(s["end"])}\n{_spk(s)}{s["text"].strip()}\n\n')

    with open(base+'.vtt','w',encoding='utf-8') as f:
        f.write('WEBVTT\n\n')
        for s in segs:
            f.write(f'{_ts(s["start"],False)} --> {_ts(s["end"],False)}\n{_spk(s)}{s["text"].strip()}\n\n')

    with open(base+'.tsv','w',encoding='utf-8') as f:
        f.write('start\tend\tspeaker\ttext\n')
        for s in segs:
            f.write(f'{int(s["start"]*1000)}\t{int(s["end"]*1000)}\t{s.get("speaker","")}\t{s["text"].strip()}\n')
    print(f'[ok] {name} 出力 -> {base}.{{json,txt,srt,vtt,tsv}}')

## 6. モデルロード (1回だけ)

In [ ]:
import whisperx, gc
device = 'cuda'
compute_type = 'float16'

asr_model = whisperx.load_model(MODEL, device, compute_type=compute_type, language=LANGUAGE)
align_model, align_meta = whisperx.load_align_model(language_code=LANGUAGE, device=device)

diarize_model = None
if USE_DIARIZATION:
    assert HF_TOKEN, 'HF_TOKEN未設定: 設定セルで入力するか USE_DIARIZATION=False'
    try:
        from whisperx.diarize import DiarizationPipeline
    except ImportError:
        from whisperx import DiarizationPipeline
    diarize_model = DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
print('models loaded')

## 7. 一括実行

In [ ]:
BATCH_SIZE = 16

for mp4 in mp4s:
    name = os.path.splitext(os.path.basename(mp4))[0]
    print(f'\n=== {name} ===')
    audio_path = preprocess(mp4)

    audio = whisperx.load_audio(audio_path)
    result = asr_model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)
    result = whisperx.align(result['segments'], align_model, align_meta, audio, device,
                            return_char_alignments=False)

    if diarize_model is not None:
        diarize_segments = diarize_model(audio)
        result = whisperx.assign_word_speakers(diarize_segments, result)

    result['language'] = LANGUAGE
    write_outputs(result, name)
    gc.collect(); torch.cuda.empty_cache()

print('\n全件完了')